# FinQA reusable preprocessing components

This notebook exercises the tested reusable preprocessing components against the pinned raw FinQA files. It preserves raw records and split assignments, writes only an ignored aggregate diagnostic summary, and does not train a model or execute reasoning programs.

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import pandas as pd

from finevidence.data import (
    FORBIDDEN_INFERENCE_FIELDS,
    INFERENCE_INPUT_FIELDS,
    assert_inference_safe,
    audit_record_annotations,
    build_structured_input,
    load_finqa_split,
    normalize_financial_value,
    parse_finqa_program,
    serialize_table_for_retrieval,
)

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
splits = {name: load_finqa_split(RAW_DIR / f"{name}.json") for name in ("train", "dev", "test")}
pd.Series({name: len(records) for name, records in splits.items()}, name="records")

train    6251
dev       883
test     1147
Name: records, dtype: int64

## 1. Reversible financial-value candidates

The normalizer reports only a surface-form candidate. It never converts to a number, infers a unit, or changes the raw string. Years, date-like values, unavailable markers, and compound forms remain explicit unresolved states.

In [2]:
adi = next(record for record in splits["train"] if record["id"] == "ADI/2009/page_49.pdf-1")
raw_examples = [adi["table"][0][0], adi["table"][0][1], adi["table"][1][1], adi["table"][1][2]]
candidate_examples = [
    normalize_financial_value(value, local_context="raw table cell") for value in raw_examples
]
display(pd.DataFrame([candidate.__dict__ for candidate in candidate_examples]))

value_statuses = {}
for split_name, records in splits.items():
    counts = Counter(
        normalize_financial_value(str(cell)).status
        for record in records
        for row in record["table"]
        for cell in row
    )
    value_statuses[split_name] = counts

value_status_frame = pd.DataFrame(value_statuses).fillna(0).astype(int)
display(value_status_frame)

,raw,status,numeric_token,sign_style,currency_symbol,percent_marker,scale_word,local_context
0,,unavailable,NaN,NaN,NaN,False,None,raw table cell
1,october 31 2009,ambiguous_date_or_year,NaN,NaN,NaN,False,None,raw table cell
2,$ 6427,recognized,6427,unsigned,$,False,None,raw table cell
3,$ -23158 ( 23158 ),unsupported,NaN,NaN,NaN,False,None,raw table cell


,train,dev,test
unavailable,3701,399,699
ambiguous_date_or_year,14016,1977,2590
unsupported,60001,8439,11146
recognized,69191,10059,13634
ambiguous_scale_only,152,21,16


## 2. Non-executing program parser

The parser produces operation and argument structure only. `program`, `program_re`, `steps`, and `exe_ans` are annotation fields and are not supplied to input builders. The selected examples below are observed train records that previously motivated grammar extensions.

In [3]:
observed_program_ids = [
    "ALLE/2016/page_29.pdf-1",
    "RSG/2008/page_114.pdf-1",
    "AWK/2014/page_122.pdf-3",
    "HOLX/2005/page_27.pdf-1",
]
program_examples = []
for record_id in observed_program_ids:
    record = next(item for item in splits["train"] if item["id"] == record_id)
    result = parse_finqa_program(record["qa"]["program"])
    program_examples.append(
        {
            "id": record_id,
            "program": record["qa"]["program"],
            "parsed": result.is_valid,
            "operations": [step.operation for step in result.steps],
            "argument_kinds": [step.argument_kinds for step in result.steps],
            "error": result.error,
        }
    )
display(pd.DataFrame(program_examples))

diagnostics = {
    split_name: [audit_record_annotations(record) for record in records]
    for split_name, records in splits.items()
}
coverage = pd.DataFrame(
    {
        split_name: {
            "records": len(items),
            "programs_parsed": sum(item.program_parse.is_valid for item in items),
            "invalid_evidence_references": sum(item.invalid_evidence_count for item in items),
            "empty_answer": sum(item.answer_is_empty for item in items),
            "empty_explanation": sum(item.explanation_is_empty for item in items),
        }
        for split_name, items in diagnostics.items()
    }
).T
display(coverage)
assert (coverage["records"] == coverage["programs_parsed"]).all()

,id,program,parsed,operations,argument_kinds,error
0,ALLE/2016/page_29.pdf-1,"table_average(2016, none)",True,[table_average],"[(table_label, none)]",None
1,RSG/2008/page_114.pdf-1,"divide(285.7, .76)",True,[divide],"[(numeric_literal, numeric_literal)]",None
2,AWK/2014/page_122.pdf-3,"multiply(3176, const_m1), divide(#0, 10379)",True,"[multiply, divide]","[(numeric_literal, named_constant), (intermedi...",None
3,HOLX/2005/page_27.pdf-1,"table_sum(17695228, 963202), add(#0, 155213)",True,"[table_sum, add]","[(numeric_literal, numeric_literal), (intermed...",None


,records,programs_parsed,invalid_evidence_references,empty_answer,empty_explanation
train,6251,6251,1,48,5276
dev,883,883,0,12,742
test,1147,1147,0,14,970


## 3. Leakage-safe input construction

The structured representation has a strict allowlist. The independent table serialization preserves the raw header, row boundaries, and header-to-cell association. It is deliberately generated from `table`, not from publisher retrieval/model artifacts.

In [4]:
sample = adi
structured = build_structured_input(sample)
table_text = serialize_table_for_retrieval(sample["table"])
assert_inference_safe(structured)

print("Allowlisted keys:", sorted(INFERENCE_INPUT_FIELDS))
print("Structured payload keys:", sorted(structured))
print("Forbidden fields excluded:", sorted(FORBIDDEN_INFERENCE_FIELDS))
print("\nFirst serialized rows:")
print("\n".join(table_text.splitlines()[:2]))

sentinel = json.loads(json.dumps(sample))
sentinel["qa"]["answer"] = "__FORBIDDEN_SENTINEL__"
sentinel_input = build_structured_input(sentinel)
assert "__FORBIDDEN_SENTINEL__" not in json.dumps(sentinel_input)
assert not (set(sentinel_input) & FORBIDDEN_INFERENCE_FIELDS)

Allowlisted keys: ['id', 'post_text', 'pre_text', 'question', 'table']
Structured payload keys: ['id', 'post_text', 'pre_text', 'question', 'table']
Forbidden fields excluded: ['ann_table_rows', 'ann_text_rows', 'answer', 'exe_ans', 'explanation', 'gold_inds', 'model_input', 'program', 'program_re', 'steps', 'table_retrieved', 'table_retrieved_all', 'text_retrieved', 'text_retrieved_all', 'tfidftopn']

First serialized rows:
header |  | october 31 2009 | november 1 2008
row_1 | : fair value of forward exchange contracts asset ( liability ) | october 31 2009: $ 6427 | november 1 2008: $ -23158 ( 23158 )


## 4. Persisted diagnostic summary

Only aggregate counts and source paths are saved under the ignored processed-data directory. Raw records and official split files are neither modified nor copied. This report distinguishes parser coverage from source-data validity: complete parser coverage does not certify the correctness of annotations.

In [5]:
summary = {
    "source": {
        name: str(path.relative_to(PROJECT_ROOT))
        for name, path in {name: RAW_DIR / f"{name}.json" for name in splits}.items()
    },
    "parser_coverage": coverage[["records", "programs_parsed"]].astype(int).to_dict(orient="index"),
    "quality_warnings": coverage[
        ["invalid_evidence_references", "empty_answer", "empty_explanation"]
    ]
    .astype(int)
    .to_dict(orient="index"),
    "value_candidate_statuses": {name: dict(counts) for name, counts in value_statuses.items()},
    "limitations": [
        "Parser coverage is not annotation validity and does not execute programs.",
        "Value candidates do not infer units, periods, or final numeric values.",
        "Allowlist checks cannot prove absence of semantic leakage in report context or unverified external artifacts.",
    ],
}
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
summary_path = PROCESSED_DIR / "m2_preprocessing_diagnostics.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(summary_path.relative_to(PROJECT_ROOT))
display(pd.DataFrame(summary["quality_warnings"]).T)

data/processed/m2_preprocessing_diagnostics.json


,invalid_evidence_references,empty_answer,empty_explanation
train,1,48,5276
dev,0,12,742
test,0,14,970


## Decision

The raw record remains authoritative. Value candidates preserve raw surface strings and unresolved states; program parsing is non-executing and reported separately from annotation quality; one unresolved `text_-1` reference remains flagged; and model inputs are rebuilt from only the allowlisted raw context.

The exact next step is to document a task-specific canonical-answer policy, then build a train-only baseline over the allowlisted input representation. Do not use `model_input`, retrieved fields, gold evidence, programs, or executable answers as answer-prediction features.